# Compare Hirriririir to Ground Truth (extended metrics)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import SimpleITK as sitk
from dissector.evaluation import binary_cross_entropy, boundary_iou_3d, inter_slice_dice

In [ ]:
MODALITY = 'WATER'   # 'FAT', 'WATER', or 'FATFRAC'

BOUNDARY_DISTANCE = 1

if MODALITY == 'FATFRAC':
    SEG_DIR = os.path.join('..', 'segmentations_fat_frac')
else:
    SEG_DIR = os.path.join('..', f'segs_{MODALITY.lower()}')

GT_BASE    = os.path.join('..', '..', 'myosegmenTUM')
RESULT_DIR = os.path.join('..', f'results/results_{MODALITY.lower()}')
os.makedirs(RESULT_DIR, exist_ok=True)

HIRR_GRACILIS  = 7
HIRR_SARTORIUS = 1
GT_L_GRACILIS  = 1
GT_R_GRACILIS  = 5
GT_L_SARTORIUS = 4
GT_R_SARTORIUS = 8

def stem_to_subject(stem):
    # works for HV001_1_FAT_stack1, HV001_1_WATER_stack1, HV001_1_FATFRACTION_stack1
    return re.split(r'_(FAT|WATER|FATFRACTION)_', stem)[0]

seg_files = sorted(f for f in os.listdir(SEG_DIR) if f.endswith('_thigh_seg.nii.gz'))
print(f'Modality  : {MODALITY}')
print(f'Seg dir   : {os.path.abspath(SEG_DIR)}')
print(f'Result dir: {os.path.abspath(RESULT_DIR)}')
print(f'Found     : {len(seg_files)} segmentation files')

## Evaluate all muscles

In [ ]:
MUSCLES = [
    ('R_gracilis',  GT_R_GRACILIS,  HIRR_GRACILIS),
    ('L_gracilis',  GT_L_GRACILIS,  HIRR_GRACILIS),
    ('R_sartorius', GT_R_SARTORIUS, HIRR_SARTORIUS),
    ('L_sartorius', GT_L_SARTORIUS, HIRR_SARTORIUS),
]

def evaluate_muscle(muscle_name, gt_label_idx, pred_label_idx):
    results = []
    for seg_file in seg_files:
        stem      = seg_file.replace('_thigh_seg.nii.gz', '')
        subject   = stem_to_subject(stem)
        m         = re.search(r'stack(\d+)', stem)
        if not m:
            print(f'  could not parse stack number: {seg_file}, skipping')
            continue
        stack_num = m.group(1)
        gt_name   = os.path.join(GT_BASE, subject, 'SegmentationMasks',
                                 f'combined_gt_stack{stack_num}.mha')
        if not os.path.exists(gt_name):
            print(f'  GT not found: {gt_name}, skipping')
            continue

        gt_image   = sitk.ReadImage(gt_name)
        pred_image = sitk.ReadImage(os.path.join(SEG_DIR, seg_file))

        gt     = sitk.Cast(gt_image == gt_label_idx, sitk.sitkUInt8)
        gt_arr = sitk.GetArrayFromImage(gt).astype(float)

        pred_arr  = sitk.GetArrayFromImage(
            sitk.Cast(pred_image == pred_label_idx, sitk.sitkUInt8)).astype(float)
        pred_sitk = sitk.GetImageFromArray(pred_arr.astype(np.uint8))
        pred_sitk.CopyInformation(gt_image)
        pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

        dice_filter = sitk.LabelOverlapMeasuresImageFilter()
        dice_filter.Execute(gt, pred)

        if gt_arr.sum() > 0 and pred_arr.sum() > 0:
            hd_filter = sitk.HausdorffDistanceImageFilter()
            hd_filter.Execute(gt, pred)
            hd = hd_filter.GetHausdorffDistance()
        else:
            print(f'  {seg_file}: empty mask (gt={int(gt_arr.sum())} pred={int(pred_arr.sum())}), HD=NaN')
            hd = np.nan

        results.append({
            'image':                                gt_name,
            'pred_label':                           seg_file,
            f'{muscle_name}_dice':                  dice_filter.GetDiceCoefficient(),
            f'{muscle_name}_hausdorff':             hd,
            f'{muscle_name}_jaccard':               dice_filter.GetJaccardCoefficient(),
            f'{muscle_name}_volume_similarity':     dice_filter.GetVolumeSimilarity(),
            f'{muscle_name}_false_negative':        dice_filter.GetFalseNegativeError(),
            f'{muscle_name}_false_positive':        dice_filter.GetFalsePositiveError(),
            f'{muscle_name}_bce':                   binary_cross_entropy(gt_arr, pred_arr),
            f'{muscle_name}_boundary_iou_3d':       boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr),
            f'{muscle_name}_inter_slice_dice_pred': inter_slice_dice(pred_arr),
            f'{muscle_name}_inter_slice_dice_gt':   inter_slice_dice(gt_arr),
        })

    df = pd.DataFrame(results)
    csv_path = os.path.join(RESULT_DIR, f'df_{muscle_name}_hirriririir.csv')
    df.to_csv(csv_path)
    print(f'  Saved {len(df)} rows → {csv_path}')
    return df

dfs = {}
for muscle_name, gt_idx, pred_idx in MUSCLES:
    print(f'\n── {muscle_name} ──')
    dfs[muscle_name] = evaluate_muscle(muscle_name, gt_idx, pred_idx)

print('\nDone.')

## Results

In [ ]:
for name, df in dfs.items():
    print(f'\n── {name} ──')
    display(df[['pred_label', f'{name}_dice', f'{name}_hausdorff']].head())